In [ ]:
!pip install transformers datasets accelerate evaluate jiwer huggingface_hub

In [ ]:
import transformers, datasets, accelerate, evaluate, huggingface_hub
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("evaluate:", evaluate.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

transformers: 5.0.0
datasets: 4.8.3
accelerate: 1.12.0
evaluate: 0.4.6
huggingface_hub: 1.4.1


In [ ]:
# ============================================================
# 1  Imports
# ============================================================

import os
import re
import json
import torch
import pandas as pd
import numpy as np

from dataclasses import dataclass
from typing import Union

from datasets import Dataset, Audio
from evaluate import load
from sklearn.model_selection import train_test_split
from transformers import set_seed

import transformers
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,          # MMS uses Wav2Vec2ForCTC (not Conformer)
    TrainingArguments,
    Trainer,
)

print("Transformers version:", transformers.__version__)
set_seed(42)

# ============================================================
# 2  Paths  (edit if your layout differs)
# ============================================================

DATA_DIR   = "/kaggle/input/datasets/buumbachinjila/zambezi-voice-speech-corpus/bemba"
AUDIO_DIR  = os.path.join(DATA_DIR, "audio/")
TRAIN_TSV  = os.path.join(DATA_DIR, "/kaggle/input/datasets/buumbachinjila/zambezi-voice-speech-corpus/bemba/transcripts/train.csv")
OUTPUT_DIR = "./bemba-mms-model"
FINAL_DIR  = "./bemba-mms-final"
VOCAB_FILE = "vocab.json"


Transformers version: 5.0.0


In [ ]:
pd.read_csv(TRAIN_TSV, sep=",").head()

,audio,duration,sentence
0,09-200916-052004_bem_a40_elicit_0.wav,4.061063,nde fwaya uku samba cino icungulo
1,09-200916-052004_bem_a40_elicit_2.wav,4.038375,nde fwaya ukuafwa ukulima ukusambilisha pali z...
2,09-200916-052004_bem_a40_elicit_3.wav,2.404875,shinga ubwali ne inkoko
3,09-200916-052004_bem_a40_elicit_4.wav,3.357750,mailo akasuba nkafwaya ukusalula imbalala
4,09-200916-052004_bem_a40_elicit_5.wav,4.015688,nde umfwa sana utulo nka samba mailo ulucelo


In [ ]:

# ============================================================
# 3  Load Dataset
# ============================================================

print("Loading Bemba dataset...")

df = pd.read_csv(TRAIN_TSV, sep=",")
df = df[["audio", "sentence"]].rename(columns={"sentence": "text"})
df["path"] = AUDIO_DIR + df["audio"]
df = df.dropna()

total_before = len(df)
df["exists"]   = df["path"].apply(os.path.exists)
found_df       = df[df["exists"]]
missing_df     = df[~df["exists"]]

print(f"Total rows in TSV (after dropna): {total_before}")
print(f"Audio files FOUND:   {len(found_df)}")
print(f"Audio files MISSING: {len(missing_df)}")

if len(missing_df) > 0:
    print(f"Missing rate: {len(missing_df)/total_before*100:.1f}%")
    for _, row in missing_df.head(10).iterrows():
        print(f"  {row['path']}")

df = df[df["exists"]][["path", "text"]].reset_index(drop=True)
print(f"\nUsable samples: {len(df)}")


Loading Bemba dataset...
Total rows in TSV (after dropna): 11906
Audio files FOUND:   11906
Audio files MISSING: 0

Usable samples: 11906


In [ ]:
# ============================================================
# 4  Train / Validation Split
# ============================================================

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))

# ============================================================
# 5  HuggingFace Dataset
# ============================================================

def create_dataset(df):
    dataset = Dataset.from_pandas(df)
    dataset = dataset.cast_column("path", Audio(sampling_rate=16000))
    return dataset

train_dataset = create_dataset(train_df)
val_dataset   = create_dataset(val_df)

# ============================================================
# 6  Clean Text  (Tonga-specific)
# ============================================================

CHARS_TO_REMOVE   = r'[\,\?\.\!\-\;\:\"\"%\'\"\'\"\'…\(\)\[\]\{\}]'
NON_TONGA_LETTERS = r'[qx]'   #Tonga does NOT use q or x

def clean_text(batch):
    text = batch["text"].lower()
    text = re.sub(CHARS_TO_REMOVE, "", text)
    text = re.sub(r"[0-9]", "", text)
    text = re.sub(NON_TONGA_LETTERS, "", text)
    text = re.sub(r"[^\x00-\x7F]", "", text)   # strip non-ASCII
    text = re.sub(r" +", " ", text).strip()
    batch["text"] = text
    return batch

train_dataset = train_dataset.map(clean_text)
val_dataset   = val_dataset.map(clean_text)

train_dataset = train_dataset.filter(lambda x: len(x["text"].strip()) > 0)
val_dataset   = val_dataset.filter(lambda x: len(x["text"].strip()) > 0)

print("\n--- Sample cleaned text ---")
for i in range(5):
    print(repr(train_dataset[i]["text"]))
print("---\n")

# ============================================================
# 7  Build Vocabulary  (Tonga-specific, 27 chars)
# ============================================================

def extract_chars(batch):
    all_text = " ".join(batch["text"])
    return {"vocab": [list(set(all_text))]}

vocab_dataset = train_dataset.map(
    extract_chars,
    batched=True,
    batch_size=-1,
    remove_columns=train_dataset.column_names,
)

vocab_list = sorted(set(vocab_dataset["vocab"][0]))
vocab_dict = {v: i for i, v in enumerate(vocab_list)}

if " " in vocab_dict:
    vocab_dict["|"] = vocab_dict[" "]
    del vocab_dict[" "]

for bad_char in ["q", "x"]:
    vocab_dict.pop(bad_char, None)

vocab_dict = {char: idx for idx, char in enumerate(sorted(vocab_dict.keys()))}
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

print("Vocabulary size:", len(vocab_dict))
print("Vocabulary:", sorted(vocab_dict.keys()))

assert "q" not in vocab_dict, "ERROR: 'q' found in vocabulary!"
assert "x" not in vocab_dict, "ERROR: 'x' found in vocabulary!"
print("✓ Confirmed: 'q' and 'x' are NOT in vocabulary")

with open(VOCAB_FILE, "w") as f:
    json.dump(vocab_dict, f)

# ============================================================
# 8  Processor
# ============================================================

tokenizer = Wav2Vec2CTCTokenizer(
    VOCAB_FILE,
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

# ============================================================
# 9  Prepare Dataset
# ============================================================

def prepare_dataset(batch):
    audio = batch["path"]
    batch["input_values"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    ).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names)
val_dataset   = val_dataset.map(prepare_dataset,   remove_columns=val_dataset.column_names)

# ============================================================
# 10  Filter Audio  (1 s – 30 s)
# ============================================================

MIN_LEN = 1  * 16_000
MAX_LEN = 30 * 16_000

train_dataset = train_dataset.filter(lambda x: MIN_LEN < x["input_length"] < MAX_LEN)
val_dataset   = val_dataset.filter(  lambda x: MIN_LEN < x["input_length"] < MAX_LEN)

print("Train after filtering:", len(train_dataset))
print("Validation after filtering:", len(val_dataset))

# ============================================================
# 11  Data Collator
# ============================================================

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]}          for f in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor)



Train: 10715
Validation: 1191


Map:   0%|          | 0/10715 [00:00<?, ? examples/s]

Map:   0%|          | 0/1191 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10715 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1191 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/10715 [00:00<?, ? examples/s]

Map:   0%|          | 0/1191 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10715 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1191 [00:00<?, ? examples/s]

Train after filtering: 10708
Validation after filtering: 1190


In [ ]:
# ============================================================
# 12  Load MMS Model
#
#     facebook/mms-300m
#     - Meta Massively Multilingual Speech model
#     - Pre-trained on 1000+ languages including many Bantu
#       languages
#     - ignore_mismatched_sizes=True drops the MMS vocab head
#       and initialises a fresh Tonga head (27 chars)
#
#     Other MMS options if you want to experiment:
#       facebook/mms-1b          (1B params, best quality, slow)
#       facebook/mms-1b-fl102    (fine-tuned on 102 languages)
#       facebook/mms-1b-all      (fine-tuned on 1162 languages)
# ============================================================

MODEL_NAME = "facebook/mms-300m"

model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.1,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True,   # drops MMS vocab head, adds fresh Tonga head
)

model.freeze_feature_encoder()
print(f"Model loaded: {MODEL_NAME}")
print(f"Total parameters: {model.num_parameters():,}")

# ============================================================
# 13  Training Arguments
#     Lower LR (1e-4) and more epochs (5)
#     for MMS fine-tuning on low-resource languages
# ============================================================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    eval_steps=200,
    logging_steps=100,
    save_steps=200,
    save_strategy="steps",
    num_train_epochs=9,        # MMS benefits from more epochs
    learning_rate=1e-4,         # lower LR suits MMS fine-tuning
    warmup_steps=500,
    weight_decay=0.005,
    gradient_checkpointing=True,
    fp16=True,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",
    length_column_name="input_length",
)

# ============================================================
# 14  WER Metric
# ============================================================

wer_metric = load("wer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids    = np.argmax(pred_logits, axis=-1)
    pred_str    = processor.batch_decode(pred_ids)

    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

# ============================================================
# 15  Trainer
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=processor,
    compute_metrics=compute_metrics,
)



config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/mms-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: facebook/mms-300m
Total parameters: 315,468,445


In [ ]:
# ============================================================
# 16  Train
# ============================================================

print("\n========== STARTING TRAINING ==========")
print(f"Model                : {MODEL_NAME}")
print(f"Transformers version : {transformers.__version__}")
print(f"Tonga vocabulary     : {len(processor.tokenizer)} characters")
print(f"Train samples        : {len(train_dataset)}")
print(f"Val samples          : {len(val_dataset)}")
print(f"Epochs               : {training_args.num_train_epochs}")
print(f"Effective batch size : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate        : {training_args.learning_rate}")
print("========================================\n")

trainer.train()

# ============================================================
# 17  Save Model & Processor
# ============================================================

model.save_pretrained(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
print(f"Model and processor saved to {FINAL_DIR}")

# ============================================================
# 18  Final Evaluation
# ============================================================

print("\n========== FINAL EVALUATION ==========")
eval_results = trainer.evaluate()
print(f"Final WER: {eval_results['eval_wer']:.4f}  ({eval_results['eval_wer']*100:.2f}%)")

# ============================================================
# 19  Test Inference  (spot-check 5 validation samples)
# ============================================================

def transcribe(idx):
    example      = val_dataset[idx]
    device       = model.device
    input_values = torch.tensor(example["input_values"]).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_values).logits

    pred_ids   = torch.argmax(logits, dim=-1)
    prediction = processor.decode(pred_ids[0])

    label_ids = [i for i in example["labels"] if i != -100]
    reference = processor.decode(label_ids)

    print(f"\n--- Sample {idx} ---")
    print("REFERENCE :", reference)
    print("PREDICTION:", prediction)

print("\n========== TEST INFERENCE ==========")
for i in range(5):
    transcribe(i)

print("\n========== TRAINING PIPELINE COMPLETE ==========")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 28, 'bos_token_id': 27}.



========== STARTING TRAINING ==========
Model                : facebook/mms-300m
Transformers version : 5.0.0
Tonga vocabulary     : 29 characters
Train samples        : 10708
Val samples          : 1190
Epochs               : 9
Effective batch size : 8
Learning rate        : 0.0001



/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss,Wer
200,13.875394,6.091068,1.000000
400,11.533408,5.701941,1.000000
600,4.337866,1.320140,0.735509
800,2.731381,0.931119,0.610754
1000,2.420869,0.829172,0.550562
1200,2.091014,0.699243,0.515605
1400,1.942587,0.658371,0.490993
1600,1.795376,0.638542,0.472178
1800,1.631597,0.612371,0.472356
2000,1.680174,0.600859,0.463082


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and processor saved to ./bemba-mms-final

========== FINAL EVALUATION ==========


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Final WER: 0.3680  (36.80%)

========== TEST INFERENCE ==========

--- Sample 0 ---
REFERENCE : joni ati kanshi shalikapo abene nganda tuleya
PREDICTION: joni ati kanshi halikapo abene ba nganda tuleya

--- Sample 1 ---
REFERENCE : ukwabula ukusambilila mu mitundu imbi
PREDICTION: ukwabula ukusambilila mu mitundu imbi

--- Sample 2 ---
REFERENCE : imfumu yafulwa nganshi yatuma abakuyaipaya mulenga
PREDICTION: imfumu yafulwa nganshi yatuma abakuyaipaya mulenga

--- Sample 3 ---
REFERENCE : mu kusondwelela ukwendauka na mafuta ukulasuba bantu amayanda imyotoka ifyakulya ne fintu fimbi tacilefwaikwa kabili cilelanga ukufuma mwisambililo lya mu cipingo cipya
PREDICTION: mukusondolwela ukwendauka na mafuta ukulasuba abantu amayanda miyotoka fya kulya ne fintu fimbi tacilefwaiko kabili cilelanga ukufuma mwisambililo lya mu cipingo ci

--- Sample 4 ---
REFERENCE : kuti bamfubu ukwisabuka ku bukali bwa mulilo bamona fye bali mu wingi
PREDICTION: kuti bamufubu ukwisabuka ku bukali bwa mulilo pa